In [1]:
# ! ck2yaml --input=USCMechII.inp --thermo=USCMechII_therm.dat --transport=USCMechII_tran.dat --permissive

In [2]:
import cantera as ct
import numpy as np
import heapq
from typing import List, Dict, Tuple, Set, Optional, Sequence
ct.suppress_deprecation_warnings()
ct.suppress_thermo_warnings()

In [ ]:
def drg_overall_importance(
    R: np.ndarray,
    target_indices: List[int],
    eps_edge: float = 1e-16,
) -> np.ndarray:
    """
    Compute DRGEP overall importance for each species based on aggregated
    DRGEP matrix R and a set of target species.

    For each target T and species A, define:

        R_TA = max over all paths T -> ... -> A of product(r_ij along path)

    Then the importance of species A is:

        I_A = max_T R_TA.

    Implementation:
      - Use weights w_ij = -log(r_ij) on edges (non-negative).
      - For each target T, run Dijkstra on w_ij.
      - distance d_TA = sum of w_ij along cheapest path.
      - R_TA = exp(-d_TA). If A is unreachable, R_TA = 0.

    Parameters
    ----------
    R : ndarray
        DRGEP matrix, shape (nsp, nsp), entries in [0,1].
    target_indices : list of int
        Species indices (global) used as targets.
    eps_edge : float
        Edges with r_ij <= eps_edge are treated as absent.

    Returns
    -------
    importance : ndarray
        DRGEP overall importance, shape (nsp,), in [0,1].
    """
    nsp = R.shape[0]
    importance = np.zeros(nsp, dtype=float)

    # Build adjacency lists with weights w_ij = -log(r_ij)
    adj = [[] for _ in range(nsp)]
    for i in range(nsp):
        for j in range(nsp):
            rij = R[i, j]
            if rij > eps_edge:
                w_ij = -np.log(rij)
                adj[i].append((j, w_ij))

    INF = 1e300

    for T in target_indices:
        # Dijkstra from target T
        dist = np.full(nsp, INF, dtype=float)
        dist[T] = 0.0
        heap = [(0.0, T)]

        while heap:
            d_u, u = heapq.heappop(heap)
            if d_u > dist[u]:
                continue
            for v, w_uv in adj[u]:
                nd = d_u + w_uv
                if nd < dist[v]:
                    dist[v] = nd
                    heapq.heappush(heap, (nd, v))

        # Convert distances to path coefficients R_TA
        with np.errstate(over="ignore", under="ignore"):
            R_TA = np.where(dist < INF, np.exp(-dist), 0.0)

        # Combine over targets: I_A = max_T R_TA
        importance = np.maximum(importance, R_TA)

    return importance

def sensitivity_mechanism_rank_from_rAB(
    mech_yaml: str,
    r_AB_global: np.ndarray,
    target_species: List[str],
    verbose: bool = False,
):
    """
    Ranking driver using a *precomputed* sensitivity-based r_AB matrix.

    Parameters
    ----------
    mech_yaml : str
        Path to mechanism yaml (full detailed mechanism).
    r_AB_global : ndarray
        Sensitivity-based matrix r_AB, shape (nsp, nsp).
        Entry [A,B] is the influence of B on A (your new definition).
    target_species : list of str
        Names of target species for DRGEP-style path importance.
    verbose : bool
        If True, print summary and top important species.

    Returns
    -------
    gas : ct.Solution
        Full mechanism solution object.
    species_names : list of str
        Names of species in the mechanism (index-aligned with r_AB_global).
    importance : ndarray, shape (nsp,)
        Global importance I_A in [0,1].
    ranked_species : list of (name, importance)
        Sorted by importance, least -> most important.
    """
    gas = ct.Solution(mech_yaml)
    species_names = gas.species_names
    nsp = gas.n_species

    r_AB_global = np.asarray(r_AB_global, dtype=float)
    if r_AB_global.shape != (nsp, nsp):
        raise ValueError(
            f"r_AB_global shape {r_AB_global.shape} does not match n_species={nsp}."
        )

    # Map target species to indices (ignore missing targets with a warning)
    target_indices = []
    for name in target_species:
        if name in species_names:
            target_indices.append(gas.species_index(name))
        else:
            print(f"[WARN] Target species '{name}' not in mechanism; ignoring.")

    if verbose:
        print("=== Sensitivity-based r_AB ranking ===")
        print(f"Mechanism: {mech_yaml}")
        print(f"Number of species: {nsp}")
        print(f"Targets: {target_species}")
        print(f"Valid target indices: {target_indices}")

    if not target_indices:
        importance = np.zeros(nsp, dtype=float)
    else:
        # Reuse the DRGEP path-importance code, but with your r_AB
        importance = drg_overall_importance(r_AB_global, target_indices)

    ranked_species = [
        (name, float(importance[i]))
        for i, name in enumerate(species_names)
    ]
    ranked_species.sort(key=lambda x: x[1])  # least -> most important

    if verbose:
        print("\nTop 15 most important species (by sensitivity-based importance):")
        for name, val in ranked_species[-15:]:
            print(f"  {name:15s}  I = {val:.4e}")

    return gas, species_names, importance, ranked_species

mech = "chem_1201fs.yaml"
r_AB_global = np.load('hehe_vectorize_fs_atj_abs_r_AB_multi_S.npy')


In [4]:
# === Sensitivity-based ranking ===
gas_sens, species_names_sens, global_imp_sens, ranked_species_sens = \
    sensitivity_mechanism_rank_from_rAB(
        mech_yaml=mech,   # 'mech' is already defined in your notebook as "chem_1111.yaml"
        r_AB_global=r_AB_global,
        target_species=["XC12H26", "HMN", "O2", "OH", "HO2", "IC4H8", "CO2", "H2O"],
        verbose=True,
    )

=== Sensitivity-based r_AB ranking ===
Mechanism: chem_1201fs.yaml
Number of species: 1833
Targets: ['XC12H26', 'HMN', 'O2', 'OH', 'HO2', 'IC4H8', 'CO2', 'H2O']
Valid target indices: [1051, 1048, 34, 37, 40, 261, 42, 36]

Top 15 most important species (by sensitivity-based importance):
  TC4H9            I = 4.9788e-02
  H                I = 7.6942e-02
  CC15H31          I = 1.9067e-01
  CH3              I = 3.3257e-01
  H2               I = 4.8410e-01
  CO               I = 4.9662e-01
  HOCHO            I = 9.6286e-01
  O2               I = 1.0000e+00
  H2O              I = 1.0000e+00
  OH               I = 1.0000e+00
  HO2              I = 1.0000e+00
  CO2              I = 1.0000e+00
  IC4H8            I = 1.0000e+00
  HMN              I = 1.0000e+00
  XC12H26          I = 1.0000e+00


In [5]:
for sp in ranked_species_sens:
  print(sp[0], sp[1])

AR 0.0
N2 0.0
HE 0.0
IC10H19CHO 0.0
C5D2Y1 0.0
C6D2Y1 0.0
C14H11O-1O2H 3.9128017085127894e-25
C14H13OO 8.190774397367655e-24
C14H12OOH 2.12565911037991e-21
C14H12O2H-1O2 2.1404067909539674e-21
IC5KETDAO 9.140281188919758e-17
IC5KETADO 1.3602643245041325e-16
NC3H7COC2H4P 3.1612945867259743e-16
C5H11O-1 4.334908160000534e-16
C3H6COC2H5-1 7.4028289960367435e-16
C14H12O 7.459486051795915e-16
C3H6COC2H5-3 9.520040349830532e-16
IC5KETCDO 1.0114162058273264e-15
NC5H10CHO-5 1.0233128328194821e-15
C14H13OOH 1.2108514261494826e-15
IC5KETDCO 1.3230399900115066e-15
C3H6COC2H5-2 1.4587764972666339e-15
NC5H10CHO-2 2.0990333421468423e-15
C6H5OOH 6.17349451735704e-15
C7H111-5,3,6P 7.799232336564593e-15
NC5H10CHO-1 9.882028392059275e-15
C7H111-5,1P 1.1969473276966552e-14
NC5H11CO 1.238484585325815e-14
IC8Y3-1R 1.7236450057578048e-14
NC3H7COC2H5 1.8505339965615353e-14
C5H10OH-5O2 1.9398911549738846e-14
NC5H10CHO-4 1.9446692706885365e-14
CH2CH2COC2H5 2.3767418907509296e-14
C5H10OH15 2.5727577829401615e-1

In [6]:
ranked_species_sens = [sp[0] for sp in ranked_species_sens]

In [7]:
def build_reduced_mechanism(
    base_mech_yaml: str,
    species_to_keep: Set[str],
    out_mech_yaml: str,
    phase_name: str = None,
    keep_original_units: bool = True,
) -> None:
    """
    Build a reduced mechanism file (YAML) keeping only species_to_keep
    and all reactions that are valid with those species.

    Parameters
    ----------
    base_mech_yaml : str
        Path to original mechanism.
    species_to_keep : set[str]
        Species names that must remain in the reduced mechanism.
    out_mech_yaml : str
        Path of the reduced YAML to write.
    phase_name : str, optional
        Phase name in base_mech_yaml; if None, use default.
    keep_original_units : bool
        If True, try to keep the same units style as the original
        (cm, mol, cal/mol) in the written YAML. If False, write in
        Cantera's native (m, kmol, J/kmol).
    """
    # 1) Read original phase to get units + species order
    gas_orig = ct.Solution(base_mech_yaml, phase_name) if phase_name else ct.Solution(base_mech_yaml)

    all_species = ct.Species.list_from_file(base_mech_yaml)
    kept_species = [sp for sp in all_species if sp.name in species_to_keep]

    if not kept_species:
        raise RuntimeError("No species left after filtering.")

    # 2) Temporary phase for reading reactions that fit kept_species
    tmp_gas = ct.Solution(
        thermo=gas_orig.thermo_model,
        kinetics="gas",        # gas-phase kinetics
        species=kept_species,
    )

    # 3) Reactions from file that are valid with tmp_gas
    all_reactions = ct.Reaction.list_from_file(base_mech_yaml, tmp_gas)

    # 4) Final reduced phase
    red_gas = ct.Solution(
        thermo=gas_orig.thermo_model,
        kinetics="gas",
        species=kept_species,
        reactions=all_reactions,
    )
    red_gas.name = "reduced_mech"

    # 5) Decide units block
    if keep_original_units and hasattr(gas_orig, "standard_concentration_units"):
        # Try to infer from original YAML style (common for GRI / USC)
        units = {
            "length": "cm",
            "time": "s",
            "quantity": "mol",
            "activation-energy": "cal/mol",
        }
        red_gas.write_yaml(out_mech_yaml, units=units)
    else:
        red_gas.write_yaml(out_mech_yaml)


def select_species_to_remove(
    ranked_species: Sequence[str],
    n_remove: int,
    unremovable: Set[str],
) -> Tuple[List[str], List[str]]:
    """
    Given a ranking from LEAST to MOST important species, pick which ones
    to remove, skipping unremovable species.

    Parameters
    ----------
    ranked_species : sequence[str]
        Species ordered from least important [0] to most important [-1].
    n_remove : int
        Desired number of species to remove.
    unremovable : set[str]
        Species that are never allowed to be removed (fuel, N2, O2, etc.).

    Returns
    -------
    to_remove : list[str]
        Species selected for removal (up to n_remove elements).
    skipped : list[str]
        Unremovable species that were encountered among the least-important
        part of the ranking.
    """
    to_remove = []
    skipped = []

    for sp in ranked_species:
        if sp in unremovable:
            skipped.append(sp)
            continue
        if sp in to_remove:
            continue
        to_remove.append(sp)
        if len(to_remove) >= n_remove:
            break

    # if len(to_remove) < n_remove:
    #     print(
    #         f"[select_species_to_remove] Requested removal of {n_remove} species, "
    #         f"but only {len(to_remove)} removable species were available."
    #     )

    return to_remove, skipped


def build_reduced_mechanism_from_rank(
    base_mech_yaml: str,
    ranked_species: Sequence[str],
    n_remove: int,
    unremovable: Set[str],
    out_mech_yaml: str,
    phase_name: str = None,
    keep_original_units: bool = True,
) -> Tuple[List[str], List[str]]:
    """
    Use a ranking (least -> most important) to build a reduced mechanism
    where the N least-important *removable* species are removed.

    Parameters
    ----------
    base_mech_yaml : str
        Path to original mechanism YAML.
    ranked_species : sequence[str]
        Species names ordered from least to most important.
    n_remove : int
        How many species you want to remove.
    unremovable : set[str]
        Species that must never be removed.
    out_mech_yaml : str
        Path/name for the reduced mechanism YAML.
    phase_name : str, optional
        Phase name in base_mech_yaml (e.g. "gas"); if None, default.
    keep_original_units : bool
        Whether to write Arrhenius parameters in Chemkin-style units.

    Returns
    -------
    removed_species : list[str]
        Species actually removed.
    skipped_unremovable : list[str]
        Unremovable species that appeared near the bottom of the ranking.
    """
    # Load original gas to get the full species set
    gas_orig = ct.Solution(base_mech_yaml, phase_name) if phase_name else ct.Solution(base_mech_yaml)
    all_species_set = set(gas_orig.species_names)

    # 1) Choose which species to remove based on ranking + unremovable
    to_remove, skipped = select_species_to_remove(
        ranked_species=ranked_species,
        n_remove=n_remove,
        unremovable=unremovable,
    )

    # Make sure we're not removing any species that don't exist in the mechanism
    to_remove = [sp for sp in to_remove if sp in all_species_set]
    removed_set = set(to_remove)

    # 2) Determine species to keep
    species_to_keep = all_species_set - removed_set

    # Always ensure unremovable are in the kept set
    species_to_keep |= unremovable & all_species_set

    if not species_to_keep:
        raise RuntimeError("No species left to keep after filtering!")

    # 3) Build the reduced mechanism
    build_reduced_mechanism(
        base_mech_yaml=base_mech_yaml,
        species_to_keep=species_to_keep,
        out_mech_yaml=out_mech_yaml,
        phase_name=phase_name,
        keep_original_units=keep_original_units,
    )

    # print(f"[build_reduced_mechanism_from_rank] Removed {len(to_remove)} species.")
    # if skipped:
    #     print(f"[build_reduced_mechanism_from_rank] Skipped {len(skipped)} unremovable species.")

    return to_remove, skipped


In [8]:
def load_pressure_profile(filename):
    """
    Returns:
      t_prof   [s]   : array of times
      p_rel    [-]   : array of p/p0 values
    """
    # skip 3 comment lines + 1 header line "<time> <p/p0>"
    data = np.loadtxt(filename, comments='!', skiprows=4)
    t_prof = data[:, 0]
    p_rel  = data[:, 1]
    return t_prof, p_rel

def build_volume_history_from_pressure(
    gas: ct.Solution,
    T0: float,
    P0: float,
    X0: str,
    t_prof: np.ndarray,
    p_rel: np.ndarray,
):
    """
    From the experimental p(t)/p0 profile, build an isentropic volume
    history V(t) and its derivative dV/dt.

    Assumptions:
      - ideal gas, adiabatic, isentropic compression
      - mixture composition frozen for the facility compression
    """
    # Initial post-reflected-shock state (reactive mixture)
    gas.TPX = T0, P0, X0

    rho0 = gas.density  # kg/m^3
    V0 = 1.0 / rho0     # m^3 for 1 kg of mixture

    # Effective gamma
    cp0 = gas.cp_mass
    cv0 = gas.cv_mass
    gamma = cp0 / cv0

    # Absolute pressure profile
    P_prof = P0 * p_rel

    # Isentropic V(t)
    V_prof = V0 * (P0 / P_prof)**(1.0 / gamma)

    # dV/dt for the moving wall
    dVdt_prof = np.gradient(V_prof, t_prof)

    return V0, t_prof, V_prof, dVdt_prof

def run_idt_with_pressure_profile(
    gas: ct.Solution,
    T0: float,
    P0: float,
    X0: str,
    t_prof: np.ndarray,
    p_rel: np.ndarray,
    reference_species: str = "CHV",
    t_end: float | None = None,
):
    """
    Integrate ignition in a shock-tube reactor using an equivalent
    volume history derived from the experimental pressure profile.

    Returns:
      tau_ign [s]
      time_history [np.ndarray]
      ref_history [np.ndarray] (mole fraction of reference species)
    """
    # Build volume history from p(t)
    V0, t_tab, V_tab, dVdt_tab = build_volume_history_from_pressure(
        gas, T0, P0, X0, t_prof, p_rel
    )

    # Initial reactive state
    gas.TPX = T0, P0, X0

    # Reactor and environment
    r = ct.IdealGasMoleReactor(gas, name="ShockTubeReactor")
    r.volume = V0

    # Environment reservoir (composition doesn't matter; no heat transfer)
    env = ct.Reservoir(gas)

    # Moving wall: V̇(t) = A * v(t), choose A = 1, so v(t) = dV/dt
    def wall_velocity(t):
        return float(np.interp(t, t_tab, dVdt_tab))

    w = ct.Wall(
        left=r,
        right=env,
        A=1.0,
        K=0.0,  # no pressure-driven motion
        U=0.0,  # adiabatic wall
        velocity=wall_velocity
    )

    net = ct.ReactorNet([r])
    net.preconditioner = ct.AdaptivePreconditioner()
    net.atol = 1e-12
    net.rtol = 1e-6

    if t_end is None:
        # By default, simulate over full profile;
        # ignition should happen well before this.
        t_end = t_tab[-1]

    t_hist = [0.0]
    T_hist = [gas.T]
    P_hist = [gas.P] 
    Y_hist = [gas.Y]
    ref_hist = [0.0]
    OH_hist = [0.0]
    temp_storage = []
    while net.time < 0.1:
        Y_dict = gas.mass_fraction_dict()
        state_dict = {
            "T": gas.T,          # temperature [K]
            "P": gas.P,          # pressure [Pa]; you can also store ct.one_atm if it's fixed
            "X": Y_dict          # composition as a dict
        }
        temp_storage.append(state_dict)

        t = net.step()
        t_hist.append(t)
        T_hist.append(gas.T)
        P_hist.append(gas.P)
        Y = gas.Y 
        Y = np.clip(Y, a_min=0.0, a_max=None)
        s = Y.sum()
        if s > 0:
            Y /= s
        Y_hist.append(Y)
        ref_hist.append(r.thermo[reference_species].X[0])
        OH_hist.append(r.thermo["OH"].X[0])

    t_hist = np.array(t_hist)
    T_hist = np.array(T_hist)
    P_hist = np.array(P_hist)
    Y_hist = np.array(Y_hist)
    ref_hist = np.array(ref_hist)
    OH_hist = np.array(OH_hist)

    dTdt = np.gradient(T_hist, t_hist, edge_order=2)
    dCHVdt = np.gradient(ref_hist, t_hist, edge_order=2)
    dOHdt = np.gradient(OH_hist, t_hist, edge_order=2)
    # IDT definition: time of max CH* (CHV) mole fraction
    i_ign_CHV = np.argmax(ref_hist) 
    i_ign_dTdt = np.argmax(dTdt)
    i_ign_dCHVdt = np.argmax(dCHVdt)
    i_ign_dOHdt = np.argmax(dOHdt)
    tau_ign_CHV = t_hist[i_ign_CHV]
    tau_ign_dTdt = t_hist[i_ign_dTdt]
    tau_ign_dCHVdt = t_hist[i_ign_dCHVdt]
    tau_ign_dOHdt = t_hist[i_ign_dOHdt]

    return tau_ign_CHV, tau_ign_dTdt, tau_ign_dCHVdt, tau_ign_dOHdt, t_hist, T_hist, P_hist, Y_hist, i_ign_CHV, temp_storage


In [9]:
base_mech = "chem_1201fs.yaml"
gas_orig = ct.Solution(base_mech)

In [10]:
# base_mech = "chem_1201.yaml"
# gas_orig = ct.Solution(base_mech)
ranked_species = ranked_species_sens   # your existing list, least -> most important
# n_remove = [500, 1000, 1100, 1200, 1300, 1350, 1400, 1450, 1500, 1550, 1600, 1625, 1650, 1675, 
#             1700, 1701, 1702, 1703, 1704, 1705, 1706, 1707, 1708, 1709, 1710, 1711, 1712, 1713, 1714, 1715, 1720, 1725]
# n_remove = [500, 1000, 1100, 1200, 1300, 1350, 1400, 1450, 1500, 1550, 1600, 1625, 1650, 1675,
#             1676, 1677, 1678, 1679, 1680, 1681, 1682, 1683, 1684, 1685, 1686, 1687, 1688, 1689, 1690, 
#             1691, 1692, 1693, 1694, 1695, 1696, 1697, 1698, 1699, 1700]
# n_remove = [1511, 1512, 1513, 1514, 1515, 1516, 1517, 1518, 1519]
n_remove = [500, 1000, 1100, 1200, 1300, 1350, 1400, 1450, 1500, 1512, 1619, 1627, 1628, 1645, 1646, 
            1651, 1652, 1683, 1684, 1691, 1692, 1697, 1698, 1699]
# n_remove = [1693, 1694, 1695, 1696]
unremovable = {
    "H2", "O2", "N2", "AR", "HE", "OH", "XC12H26", "HMN", "CHV"
}

T_L, T_H, inc = (840, 1740, 100)
T = np.arange(T_L, T_H + inc, inc)
P = [16e05] # Pa 
C = [{"XC12H26": 0.004233, "HMN": 0.000867, "O2": 0.099, "N2": 0.8959}, {"XC12H26": 0.0083, "HMN": 0.0017, "O2": 0.098, "N2": 0.892}] 

pressure_profile_file="SM3 - Pressure profile of shock tube measurements.dat"
t_prof, p_rel = load_pressure_profile(pressure_profile_file)

rt = 0.1 # sec

reference_species = 'CHV'

In [11]:
tau_all = []
i = 0
for T_local in T: 
    for P_local in P: 
        for C_local in C: 

            tau_ign_CHV, tau_ign_dTdt, tau_ign_dCHVdt, tau_ign_dOHdt, t_hist, T_hist, P_hist, Y_hist, i_ign_CHV, temp_storage = run_idt_with_pressure_profile(
                gas=gas_orig,
                T0=T_local,
                P0=P_local,
                X0=C_local,
                t_prof=t_prof,
                p_rel=p_rel,
                reference_species=reference_species,
                t_end=t_prof[-1]
            )
            tau_all.append(tau_ign_CHV)

            i += 1

            print(f"Finished at T: {T_local} K, P: {P_local} Pa, and phi: {C_local}. IDT = {tau_ign_CHV}")
tau_all = np.array(tau_all)
print(f"Shape check: {tau_all.shape}")
IDT_ref = tau_all

Finished at T: 840 K, P: 1600000.0 Pa, and phi: {'XC12H26': 0.004233, 'HMN': 0.000867, 'O2': 0.099, 'N2': 0.8959}. IDT = 0.020696903079997706
Finished at T: 840 K, P: 1600000.0 Pa, and phi: {'XC12H26': 0.0083, 'HMN': 0.0017, 'O2': 0.098, 'N2': 0.892}. IDT = 0.014875635584728288
Finished at T: 940 K, P: 1600000.0 Pa, and phi: {'XC12H26': 0.004233, 'HMN': 0.000867, 'O2': 0.099, 'N2': 0.8959}. IDT = 0.005005953296542906
Finished at T: 940 K, P: 1600000.0 Pa, and phi: {'XC12H26': 0.0083, 'HMN': 0.0017, 'O2': 0.098, 'N2': 0.892}. IDT = 0.0037227180883506973
Finished at T: 1040 K, P: 1600000.0 Pa, and phi: {'XC12H26': 0.004233, 'HMN': 0.000867, 'O2': 0.099, 'N2': 0.8959}. IDT = 0.0026555154521982283
Finished at T: 1040 K, P: 1600000.0 Pa, and phi: {'XC12H26': 0.0083, 'HMN': 0.0017, 'O2': 0.098, 'N2': 0.892}. IDT = 0.0022158401635684645
Finished at T: 1140 K, P: 1600000.0 Pa, and phi: {'XC12H26': 0.004233, 'HMN': 0.000867, 'O2': 0.099, 'N2': 0.8959}. IDT = 0.0012700273111430785
Finished at T:

In [12]:
for n_r in n_remove: 

    out_mech = f"atj_rm{n_r}.yaml" 

    removed, skipped = build_reduced_mechanism_from_rank(
        base_mech_yaml=base_mech,
        ranked_species=ranked_species,
        n_remove=n_r,
        unremovable=unremovable,
        out_mech_yaml=out_mech,
        phase_name=None,            # or "gas" if your phase has that name
        keep_original_units=True,   # keep cm/mol/cal style in the YAML
    )

    # print("Actually removed species:", removed)
    # print("Unremovable species encountered near bottom of ranking:", skipped)
    # print("Reduced mechanism written to:", out_mech) 

    rm0_mech  = f"atj_rm{n_r}.yaml" 
    gas_rm0  = ct.Solution(rm0_mech)

    # print("Ns orig:", gas_orig.n_species, "Ns rm0:", gas_rm0.n_species)
    # print("Nr orig:", gas_orig.n_reactions, "Nr rm0:", gas_rm0.n_reactions)



    tau_all = []
    i = 0
    for T_local in T: 
        for P_local in P: 
            for C_local in C: 

                tau_ign_CHV, tau_ign_dTdt, tau_ign_dCHVdt, tau_ign_dOHdt, t_hist, T_hist, P_hist, Y_hist, i_ign_CHV, temp_storage = run_idt_with_pressure_profile(
                    gas=gas_rm0,
                    T0=T_local,
                    P0=P_local,
                    X0=C_local,
                    t_prof=t_prof,
                    p_rel=p_rel,
                    reference_species=reference_species,
                    t_end=t_prof[-1]
                )
                tau_all.append(tau_ign_CHV)

                i += 1

                # print(f"Finished at T: {T_local} K, P: {P_local} atm, and phi: {C_local}. IDT = {tau_ign_CHV}")
    tau_all = np.array(tau_all)
    IDT_compare = tau_all

    error = np.max(np.abs(IDT_compare - IDT_ref)/IDT_ref)

    print(f"Kept species: {gas_rm0.n_total_species}, induced error: {error*100:.2f} %.")


Kept species: 1333, induced error: 0.24 %.
Kept species: 833, induced error: 1.46 %.
Kept species: 733, induced error: 2.93 %.
Kept species: 633, induced error: 3.80 %.
Kept species: 533, induced error: 5.15 %.
Kept species: 483, induced error: 6.44 %.
Kept species: 433, induced error: 6.86 %.
Kept species: 383, induced error: 5.79 %.
Kept species: 333, induced error: 6.03 %.
Kept species: 321, induced error: 13.41 %.
Kept species: 214, induced error: 19.90 %.
Kept species: 206, induced error: 20.47 %.
Kept species: 205, induced error: 20.57 %.
Kept species: 188, induced error: 23.90 %.
Kept species: 187, induced error: 24.66 %.
Kept species: 182, induced error: 14.42 %.
Kept species: 181, induced error: 16.19 %.
Kept species: 150, induced error: 16.70 %.
Kept species: 149, induced error: 16.78 %.
Kept species: 142, induced error: 18.73 %.
Kept species: 141, induced error: 18.73 %.
Kept species: 136, induced error: 49.93 %.
Kept species: 135, induced error: 49.81 %.
Kept species: 134, 